In [ ]:
import torch.nn as nn


nn.Conv2d.__module__

In [2]:
import cv2
import os

image_path = os.path.normpath(
    r"E:\TrainingFramework\nnUNet-master\DATASET\nnUNet_trained_models\Dataset611_BogePos\CustomTrainer__custom_nnUNetPlans__2d\fold_0\validation"
)

print(image_path)

names = [f for f in os.listdir(image_path) if f.endswith(".png")]
images = [os.path.join(image_path, f) for f in os.listdir(image_path)]

for i, f in enumerate(images):
    image = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    image = image * 255
    cv2.imwrite(f.replace(".png", "_viz.png"), image)
    # cv2.imshow(f'{names[i]}',image)
    # cv2.waitKey(0)

E:\TrainingFramework\nnUNet-master\DATASET\nnUNet_trained_models\Dataset611_BogePos\CustomTrainer__custom_nnUNetPlans__2d\fold_0\validation


TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'

### 推理

In [4]:
import os, json, cv2
import torch
import numpy as np
from nnunetv2.experiment_planning.experiment_planners.custom_planner.model import CustomResEncUnet_V2, make_network
from nnunetv2.preprocessing.normalization.default_normalization_schemes import ZScoreNormalization

dataset_name = "Dataset515_location"

image_path = os.path.normpath(
    rf"E:\TrainingFramework\nnUNet-master\DATASET\nnUNet_trained_models\{dataset_name}\CustomTrainer__custom_nnUNetPlans__2d\fold_0\111.jpg"
)
model_path = os.path.normpath(
    rf"E:\TrainingFramework\nnUNet-master\DATASET\nnUNet_trained_models\{dataset_name}\CustomTrainer__custom_nnUNetPlans__2d\fold_0\checkpoint_best.pth"
)
plans_json = os.path.normpath(
    rf"E:\TrainingFramework\nnUNet-master\DATASET\nnUNet_preprocessed\{dataset_name}\custom_nnUNetPlans.json"
)

# read image
image = cv2.imread(image_path, cv2.IMREAD_COLOR_RGB)

# letter box
input_size = 1024
smax = max(image.shape)
h, w, c = image.shape
mask = np.ones((smax, smax, 3), image.dtype) * 128
h_start = (smax - h) // 2
w_start = (smax - w) // 2
mask[h_start : h_start + h, w_start : w_start + w, :] = image
image = mask

image = cv2.resize(image, dsize=(input_size, input_size), interpolation=cv2.INTER_AREA)

# 标准化
with open(plans_json, "r") as f:
    plans = json.load(f)
use_mask_for_norm = plans["configurations"]["2d"]["use_mask_for_norm"]
property = plans["foreground_intensity_properties_per_channel"]
normalizer = ZScoreNormalization(False, intensityproperties=property)
norm_image = normalizer.run(image=image)

# preprocess
input = torch.from_numpy(norm_image).to(torch.float)
input = input.permute(2, 0, 1)[None]  # hwc -> 1chw

# load model weights
spacing = [1, 1]
patch_size = [1024, 1024]
min_feature_size = 4
max_numpool = 9999

model = make_network(c1=3, cls_num=2, input_feature_size=patch_size, deep_supervision=False, model_type="Plain")
chkpt = torch.load(model_path, map_location="cpu", weights_only=False)
model.load_state_dict(chkpt["network_weights"])

# inference
model.eval()
output = model(input)

# postprocess
omask = torch.argmax(output, dim=1)[0]  # hw

oimage = omask.numpy()
oimage = oimage * (255 / oimage.max())
oimage = np.astype(oimage, np.uint8)

print(oimage)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


#### 修改权重文件键名

In [ ]:
import torch
import os
import copy

pth_path = os.path.normpath(
    r"E:\TrainingFramework\nnUNet-master\DATASET\nnUNet_trained_models\Dataset512_void\CustomTrainer__custom_nnUNetResEncUNetPlans__2d\fold_0\checkpoint_bestraw.pth"
)

# seg_layers

saved_model = torch.load(pth_path, weights_only=False)
pretrained_dict = saved_model["network_weights"]

old_name = "deepsup_conv"
new_name = "seg_layers"

new_weights = {}

for k, v in pretrained_dict.items():
    if old_name in k:
        full_name = k.replace(old_name, new_name)
        new_weights[full_name] = v
    else:
        new_weights[k] = v

saved_model["network_weights"] = new_weights

pth_path2 = os.path.normpath(
    r"E:\TrainingFramework\nnUNet-master\DATASET\nnUNet_trained_models\Dataset512_void\CustomTrainer__custom_nnUNetResEncUNetPlans__2d\fold_0\checkpoint_best2.pth"
)
torch.save(saved_model, pth_path2)
...

#### 最大权重